# 06 · Re-merge and re-evaluate without bf16 rounding

Notebook 05 showed that saving in bf16 turned every task vector by about 30° (cosine 0.84) and
cost averaging a third of its share. Every score so far came from those rounded models.

Here, nothing is rounded:

1. **Build** the 3 specialists and 3 merges in **float32**, straight from the adapters (exact tv = s · B @ A).
   Same merge rules, λ and TIES density as before.
2. **Evaluate** base, specialists and merges with float32 maths, same questions, greedy decoding.
3. **Compare** scores, flipped answers and RRR with the 3 Oct bf16 run.

Output goes to `checkpoints/fp32/` and `results/fp32/`. The old `results/scores.csv` is not touched.

**Cost:** 6 float32 models × ~2 GB disk = ~12 GB (delete at the end if needed).
7 models × 3 tasks to evaluate. Float32 is roughly 2× slower than bf16 on an RTX card, so expect about 2 h on your GPU.
Every step skips what is already done, so the notebook can be stopped and re-run.

In [1]:
import sys, json, gc, shutil
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, torch, pandas as pd
from safetensors import safe_open
from safetensors.torch import save_file
from src.merge.merge import LazyWeights, ties
from src.utils.config import TASKS

BASE = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 42
METHODS = ["avg", "task_arithmetic", "ties"]
MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
SPEC = {t: ROOT / f"checkpoints/specialists/{t}/seed{SEED}" for t in TASKS}
OLD_MERGED = {m: ROOT / f"checkpoints/merged/{m}" for m in METHODS}

OUT = ROOT / "checkpoints" / "fp32"          # float32 models go here
RES = "results/fp32"                         # scores.csv + generations for this run
OLD_RES = ROOT / "results"                   # 3 Oct bf16 run
FP32 = ["model.dtype=float32"]               # eval override: load and compute in float32

def sub(m): return "self_attn" if m in MODULES[:4] else "mlp"
def wname(l, m): return f"model.layers.{l}.{sub(m)}.{m}.weight"

base = LazyWeights(BASE)
N_LAYERS = json.loads((base.path / "config.json").read_text())["num_hidden_layers"]
TARGET = {wname(l, m): (l, m) for l in range(N_LAYERS) for m in MODULES}
AD = {t: safe_open(SPEC[t] / "adapter" / "adapter_model.safetensors", "pt") for t in TASKS}
ACFG = {t: json.loads((SPEC[t] / "adapter" / "adapter_config.json").read_text()) for t in TASKS}
SCALE = {t: ACFG[t]["lora_alpha"] / ACFG[t]["r"] for t in TASKS}

def exact_tv(t, l, m):
    p = f"base_model.model.model.layers.{l}.{sub(m)}.{m}"
    return SCALE[t] * AD[t].get_tensor(p + ".lora_B.weight").float() @ AD[t].get_tensor(p + ".lora_A.weight").float()

INFO = {}
for m in METHODS:
    f = OLD_MERGED[m] / "merge_info.json"
    i = json.loads(f.read_text()) if f.exists() else {}
    INFO[m] = {"lam": i.get("lam", 1.0), "density": i.get("density", 0.2)}
print("tasks:", TASKS, "| merge settings (same as 3 Oct):", INFO)

/mnt/EAD854DBD854A799/IITB Course/Sem 7/CS769 (Optimisation in Machine Learning)/Project (Does RL-induced reasoning survive merging)/Does_RL-induced_reasoning_survive_merging/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 874.72it/s]

tasks: ['gsm8k', 'logic', 'code'] | merge settings (same as 3 Oct): {'avg': {'lam': 1.0, 'density': 0.2}, 'task_arithmetic': {'lam': 1.0, 'density': 0.2}, 'ties': {'lam': 1.0, 'density': 0.2}}


---
## 1. Build the float32 models

Same formulas as `merge.py`, but on exact task vectors and kept in float32:

| model | weights |
|---|---|
| spec-t | base + exact tv_t |
| merge-avg | base + mean(exact tvs) |
| merge-task_arithmetic | base + λ · sum(exact tvs) |
| merge-ties | base + λ · TIES(exact tvs, density) |

Every other weight (embeddings, norms, biases) is the base weight, unchanged.

In [2]:
def combine(tvs, method):
    lam, dens = INFO[method]["lam"], INFO[method]["density"]
    if method == "avg":
        return tvs.mean(0)
    if method == "task_arithmetic":
        return lam * tvs.sum(0)
    return lam * ties(tvs, dens)

def delta_for(label, l, m):
    if label.startswith("spec-"):
        return exact_tv(label[5:], l, m)
    return combine(torch.stack([exact_tv(t, l, m) for t in TASKS]), label[6:])

def build(label):
    out = OUT / label
    if (out / "model.safetensors").exists():
        print(f"[skip] {label} already built"); return out
    sd = {}
    for k in base.keys():
        w = base.get(k).float()
        if k in TARGET:
            w = w + delta_for(label, *TARGET[k])
        sd[k] = w.contiguous()
    out.mkdir(parents=True, exist_ok=True)
    save_file(sd, out / "model.safetensors", metadata={"format": "pt"})
    for f in (SPEC[TASKS[0]] / "merged").iterdir():      # config + tokenizer files
        if f.is_file() and not f.name.endswith(".safetensors") and f.name != "model.safetensors.index.json":
            shutil.copy(f, out / f.name)
    (out / "build_info.json").write_text(json.dumps({"label": label, "dtype": "float32", "from": "adapters",
                                                     "merge": INFO.get(label[6:]) if label.startswith("merge-") else None}))
    print(f"[built] {label} -> {out}")
    return out

LABELS = [f"spec-{t}" for t in TASKS] + [f"merge-{m}" for m in METHODS]
PATHS = {lab: build(lab) for lab in LABELS}

[built] spec-gsm8k -> /mnt/EAD854DBD854A799/IITB Course/Sem 7/CS769 (Optimisation in Machine Learning)/Project (Does RL-induced reasoning survive merging)/Does_RL-induced_reasoning_survive_merging/rl-merge/checkpoints/fp32/spec-gsm8k
[built] spec-logic -> /mnt/EAD854DBD854A799/IITB Course/Sem 7/CS769 (Optimisation in Machine Learning)/Project (Does RL-induced reasoning survive merging)/Does_RL-induced_reasoning_survive_merging/rl-merge/checkpoints/fp32/spec-logic
[built] spec-code -> /mnt/EAD854DBD854A799/IITB Course/Sem 7/CS769 (Optimisation in Machine Learning)/Project (Does RL-induced reasoning survive merging)/Does_RL-induced_reasoning_survive_merging/rl-merge/checkpoints/fp32/spec-code
[built] merge-avg -> /mnt/EAD854DBD854A799/IITB Course/Sem 7/CS769 (Optimisation in Machine Learning)/Project (Does RL-induced reasoning survive merging)/Does_RL-induced_reasoning_survive_merging/rl-merge/checkpoints/fp32/merge-avg
[built] merge-task_arithmetic -> /mnt/EAD854DBD854A799/IITB Course/S

### Sanity check

1. Each float32 specialist, rounded back to bf16, should match the old `merged/` almost everywhere
   (both are base + s·B@A; only the order of additions differs). Expect ≥ 99.9%.
2. Each float32 model should differ from base only in the 168 LoRA target matrices.

In [3]:
k0 = wname(N_LAYERS // 2, "q_proj")
for t in TASKS:
    new = LazyWeights(PATHS[f"spec-{t}"]).get(k0)
    old = LazyWeights(SPEC[t] / "merged").get(k0)
    same = (new.to(torch.bfloat16) == old).float().mean().item()
    print(f"spec-{t:<6} float32 rounded to bf16 == old merged/ : {same:.4%}   (dtype on disk: {new.dtype})")

for lab in LABELS:
    W = LazyWeights(PATHS[lab]); changed = [k for k in base.keys() if not torch.equal(W.get(k), base.get(k).float())]
    ok = set(changed) <= set(TARGET) and len(changed) == len(TARGET)
    print(f"{lab:<22} tensors changed: {len(changed):>3}  only LoRA targets: {ok}")

spec-gsm8k  float32 rounded to bf16 == old merged/ : 100.0000%   (dtype on disk: torch.float32)
spec-logic  float32 rounded to bf16 == old merged/ : 100.0000%   (dtype on disk: torch.float32)
spec-code   float32 rounded to bf16 == old merged/ : 100.0000%   (dtype on disk: torch.float32)
spec-gsm8k             tensors changed: 168  only LoRA targets: True
spec-logic             tensors changed: 168  only LoRA targets: True
spec-code              tensors changed: 168  only LoRA targets: True
merge-avg              tensors changed: 168  only LoRA targets: True
merge-task_arithmetic  tensors changed: 168  only LoRA targets: True
merge-ties             tensors changed: 168  only LoRA targets: True


---
## 2. Evaluate with float32 maths

Same `evaluate()` as always, with `model.dtype=float32`. The base is evaluated again too:
its weights are the same as before (Qwen ships in bf16), so **base bf16 vs base float32 shows how much the maths precision alone moves the scores.**
That is the noise floor for every comparison below.

Optional: `EVAL_ROUNDED = True` also scores the **old bf16 checkpoints** with float32 maths.
That separates "rounded weights" from "bf16 maths" exactly, at the cost of ~1.5 h more.

In [4]:
from src.eval.evaluate import evaluate
from src.eval.rrr import latest_scores, compute_rrr

EVAL_ROUNDED = False
RES_R = "results/fp32_maths_bf16_weights"

def done(res, label):
    f = ROOT / res / "scores.csv"
    s = latest_scores(f) if f.exists() else {}
    return all((label, t) in s for t in TASKS)

def run(label, path, res):
    if done(res, label):
        print(f"[skip] {label} already scored in {res}"); return
    evaluate(str(path), label, TASKS, seed=0, overrides=FP32, results=res)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

run("base", BASE, RES)
for lab in LABELS:
    run(lab, PATHS[lab], RES)

if EVAL_ROUNDED:
    old_paths = {**{f"spec-{t}": SPEC[t] / "merged" for t in TASKS}, **{f"merge-{m}": OLD_MERGED[m] for m in METHODS}}
    for lab, p in old_paths.items():
        run(lab, p, RES_R)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 809.49it/s]


[eval] base                     gsm8k      acc=0.460 format=0.17 len=287
[eval] base                     logic      acc=0.180 format=0.64 len=294
[eval] base                     code       acc=0.175 format=0.04 len=278


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1926.56it/s]


[eval] spec-gsm8k               gsm8k      acc=0.440 format=0.97 len=252
[eval] spec-gsm8k               logic      acc=0.160 format=0.81 len=297
[eval] spec-gsm8k               code       acc=0.070 format=0.62 len=254


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2006.31it/s]


...

[eval] merge-avg                gsm8k      acc=0.460 format=0.69 len=256
[eval] merge-avg                logic      acc=0.270 format=0.98 len=226
[eval] merge-avg                code       acc=0.265 format=1.00 len=99


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1794.29it/s]


[eval] merge-task_arithmetic    gsm8k      acc=0.400 format=0.99 len=208
[eval] merge-task_arithmetic    logic      acc=0.345 format=0.99 len=145
[eval] merge-task_arithmetic    code       acc=0.300 format=0.99 len=93


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1940.58it/s]


[eval] merge-ties               gsm8k      acc=0.440 format=0.97 len=241
[eval] merge-ties               logic      acc=0.360 format=0.98 len=189
[eval] merge-ties               code       acc=0.325 format=1.00 len=94


---
## 3. Compare with 3 Oct

Three tables:
- **Accuracy**: old (bf16 weights, bf16 maths) vs new (exact weights, float32 maths).
- **Flipped answers**: how many of the 200 questions changed from right to wrong or back. A ±3 point change with 30 flips is very different from one with 6.
- **RRR**: old vs new, same formula, same MIN_GAIN.

With 200 questions, one score moves by about ±3 points from chance alone.

In [5]:
old = latest_scores(OLD_RES / "scores.csv")
new = latest_scores(ROOT / RES / "scores.csv")
rnd = latest_scores(ROOT / RES_R / "scores.csv") if (ROOT / RES_R / "scores.csv").exists() else {}

rows = []
for lab in ["base"] + LABELS:
    r = {"model": lab}
    for t in TASKS:
        o, n = old.get((lab, t)), new.get((lab, t))
        r[f"{t} old"] = None if o is None else 100 * o
        r[f"{t} new"] = None if n is None else 100 * n
        r[f"{t} Δ"] = None if o is None or n is None else 100 * (n - o)
        if rnd: r[f"{t} rounded-w"] = 100 * rnd[(lab, t)] if (lab, t) in rnd else None
    rows.append(r)
ACC = pd.DataFrame(rows).set_index("model")
ACC.style.format("{:.1f}", na_rep="-")

,gsm8k old,gsm8k new,gsm8k Δ,logic old,logic new,logic Δ,code old,code new,code Δ
model,,,,,,,,,
base,43.0,46.0,3.0,18.0,18.0,0.0,19.0,17.5,-1.5
spec-gsm8k,43.5,44.0,0.5,16.0,16.0,0.0,12.5,7.0,-5.5
spec-logic,38.5,39.0,0.5,33.5,36.5,3.0,19.5,20.0,0.5
spec-code,4.5,4.5,0.0,18.0,17.0,-1.0,32.5,31.5,-1.0
merge-avg,44.0,46.0,2.0,22.5,27.0,4.5,27.5,26.5,-1.0
merge-task_arithmetic,41.5,40.0,-1.5,33.0,34.5,1.5,30.5,30.0,-0.5
merge-ties,41.0,44.0,3.0,38.5,36.0,-2.5,34.0,32.5,-1.5


In [6]:
def load_gen(res_dir, label, task):
    f = res_dir / "generations" / label / f"{task}.jsonl"
    return {json.loads(l)["id"]: json.loads(l)["correct"] for l in open(f)} if f.exists() else None

rows = []
for lab in ["base"] + LABELS:
    r = {"model": lab}
    for t in TASKS:
        a, b = load_gen(OLD_RES, lab, t), load_gen(ROOT / RES, lab, t)
        if a is None or b is None:
            r[f"{t} right→wrong"] = r[f"{t} wrong→right"] = None; continue
        ids = a.keys() & b.keys()
        r[f"{t} right→wrong"] = sum(bool(a[i]) and not b[i] for i in ids)
        r[f"{t} wrong→right"] = sum(not a[i] and bool(b[i]) for i in ids)
    rows.append(r)
FLIP = pd.DataFrame(rows).set_index("model")
FLIP

,gsm8k right→wrong,gsm8k wrong→right,logic right→wrong,logic wrong→right,code right→wrong,code wrong→right
model,,,,,,
base,7.0,13.0,21.0,21.0,8.0,5.0
spec-gsm8k,NaN,NaN,NaN,NaN,NaN,NaN
spec-logic,NaN,NaN,NaN,NaN,NaN,NaN
spec-code,NaN,NaN,NaN,NaN,NaN,NaN
merge-avg,11.0,15.0,15.0,24.0,6.0,4.0
merge-task_arithmetic,7.0,4.0,11.0,14.0,3.0,2.0
merge-ties,5.0,11.0,15.0,10.0,5.0,2.0


In [7]:
def rrr_table(s, name):
    tasks = [t for t in TASKS if ("base", t) in s and (f"spec-{t}", t) in s]
    out = []
    for m in METHODS:
        lab = f"merge-{m}"
        if not all((lab, t) in s for t in tasks): continue
        vals, macro, worst = compute_rrr(s, lab, tasks)
        out.append({"run": name, "merge": m, **{t: v for t, v in zip(tasks, vals)}, "macro": macro, "worst": worst})
    gains = {t: 100 * (s[(f"spec-{t}", t)] - s[("base", t)]) for t in tasks}
    return out, gains

r_old, g_old = rrr_table(old, "3 Oct bf16")
r_new, g_new = rrr_table(new, "float32")
print("specialist gain on own task (points):")
for t in TASKS:
    print(f"  {t:<6} old {g_old.get(t, float('nan')):+5.1f}   new {g_new.get(t, float('nan')):+5.1f}")
RRR = pd.DataFrame(r_old + r_new).set_index(["merge", "run"]).sort_index()
RRR.style.format("{:.2f}", na_rep="n/a")

specialist gain on own task (points):
  gsm8k  old  +0.5   new  -2.0
  logic  old +15.5   new +18.5
  code   old +13.5   new +14.0


How to read it:
- If **base old vs new** differs by a few points, that is what float32 maths alone does. Treat changes of that size elsewhere as noise.
- If **specialists** move more than base, the bf16 rounding of their weights mattered.
- **merge-avg** is the one to watch: notebook 05 says it carried 0.25 of each specialist on disk vs 0.35 exactly.
  If its RRR rises clearly, part of the old averaging loss was bf16, not merging.
- If RRR barely changes, bf16 rounding turned the weights but not the behaviour, and the 3 Oct conclusions stand.

---
## 4. Summary to paste back

In [8]:
print("ACCURACY (%)  old = 3 Oct bf16, new = float32 weights + maths")
print(ACC.to_string(float_format=lambda v: f"{v:.1f}", na_rep="-"))
print("\nFLIPPED ANSWERS vs 3 Oct (out of 200)")
print(FLIP.to_string())
print("\nRRR")
print(RRR.to_string(float_format=lambda v: f"{v:.2f}", na_rep="n/a"))

ACCURACY (%)  old = 3 Oct bf16, new = float32 weights + maths
                       gsm8k old  gsm8k new  gsm8k Δ  logic old  logic new  logic Δ  code old  code new  code Δ
model                                                                                                          
base                        43.0       46.0      3.0       18.0       18.0      0.0      19.0      17.5    -1.5
spec-gsm8k                  43.5       44.0      0.5       16.0       16.0      0.0      12.5       7.0    -5.5
spec-logic                  38.5       39.0      0.5       33.5       36.5      3.0      19.5      20.0     0.5
spec-code                    4.5        4.5      0.0       18.0       17.0     -1.0      32.5      31.5    -1.0
merge-avg                   44.0       46.0      2.0       22.5       27.0      4.5      27.5      26.5    -1.0
merge-task_arithmetic       41.5       40.0     -1.5       33.0       34.5      1.5      30.5      30.0    -0.5
merge-ties                  41.0       44.

---
## 5. Free the disk (optional)

The float32 models take ~12 GB. Scores and generations stay in `results/fp32/`. Rebuilding takes a few minutes.

In [ ]:
# shutil.rmtree(OUT)